In [2]:
# Cell 1 — Fix raster NODATA-mask mismatches (choose policy: prefer NODATA or prefer VALUE)

from pathlib import Path
import numpy as np

# ----------------------------
# ASC read/write helpers
# ----------------------------
def read_asc(path: Path | str):
    """Read ESRI ASCII grid. Returns (arr(float, nan for NODATA), header_lines(list[str]), meta(dict), nodata(float))."""
    path = Path(path)
    lines = path.read_text().splitlines()
    header_lines = lines[:6]
    data_lines = lines[6:]

    meta = {}
    for ln in header_lines:
        k, v = ln.split()[0], ln.split()[1]
        meta[k.lower()] = float(v)

    nrows = int(meta["nrows"])
    ncols = int(meta["ncols"])
    nodata = float(meta["nodata_value"])

    arr = np.zeros((nrows, ncols), dtype=np.float32)
    for i, ln in enumerate(data_lines):
        arr[i, :] = np.asarray(ln.split(), dtype=np.float32)

    arr[arr == nodata] = np.nan
    return arr, header_lines, meta, nodata


def write_asc(path: Path | str, arr: np.ndarray, header_lines: list[str], nodata: float, fmt: str = "{:.6g}"):
    """Write ESRI ASCII grid using original header_lines. NaNs become nodata."""
    path = Path(path)
    out = np.array(arr, dtype=np.float32, copy=True)
    out[np.isnan(out)] = nodata

    with path.open("w") as f:
        for ln in header_lines:
            f.write(ln.rstrip() + "\n")
        # Write row-by-row (space-separated)
        for r in range(out.shape[0]):
            row = " ".join(fmt.format(float(x)) for x in out[r, :])
            f.write(row + "\n")


def headers_compatible(meta_list):
    """Check basic header compatibility across rasters."""
    keys = ["ncols", "nrows", "xllcorner", "yllcorner", "cellsize", "nodata_value"]
    base = {k: meta_list[0][k] for k in keys}
    for i, m in enumerate(meta_list[1:], start=1):
        for k in keys:
            if float(m[k]) != float(base[k]):
                raise ValueError(f"Header mismatch raster#{i} key={k}: {m[k]} vs {base[k]}")


# ----------------------------
# Main fixer
# ----------------------------
def fix_raster_masks(
    raster_paths: list[Path | str],
    output_dir: Path | str,
    *,
    policy: str,                  # "prefer_nodata" or "prefer_value"
    value_source: str = "first",   # when policy="prefer_value": "first" (first non-nan) or "mean" (mean of non-nan)
    dry_run: bool = False,
):
    """
    policy:
      - "prefer_nodata": any mismatch cell becomes NODATA (nan) in ALL rasters
      - "prefer_value": fill NODATA (nan) cells using a value from non-nan rasters
          value_source:
            * "first": take first non-nan among rasters
            * "mean": take mean of non-nan among rasters
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    rasters = []
    headers = []
    metas = []
    nodatas = []

    for p in raster_paths:
        arr, header_lines, meta, nodata = read_asc(p)
        rasters.append(arr)
        headers.append(header_lines)
        metas.append(meta)
        nodatas.append(nodata)

    headers_compatible(metas)
    nodata = nodatas[0]
    H, W = rasters[0].shape
    stack = np.stack(rasters, axis=0)  # (K,H,W)

    is_nan = np.isnan(stack)
    any_nan = is_nan.any(axis=0)       # (H,W)
    all_nan = is_nan.all(axis=0)       # (H,W)
    mismatch = any_nan & (~all_nan)    # cells where some rasters are nan and some are not

    n_mismatch = int(mismatch.sum())
    print(f"Loaded {len(raster_paths)} rasters: shape={H}x{W}, NODATA={nodata}")
    print(f"Mismatch cells (some NODATA, some valid): {n_mismatch}")

    if n_mismatch == 0:
        print("No mismatches found. Nothing to do.")
        return

    # Apply policy
    fixed = stack.copy()

    if policy == "prefer_nodata":
        # Force all rasters to NODATA at mismatch cells
        fixed[:, mismatch] = np.nan

    elif policy == "prefer_value":
        # Fill NODATA rasters at mismatch cells using values from the non-NODATA rasters
        # Build a "fill value" per cell from the stack
        if value_source == "first":
            fill = np.full((H, W), np.nan, dtype=np.float32)
            # pick first non-nan along K
            for k in range(fixed.shape[0]):
                take = np.isnan(fill) & (~np.isnan(fixed[k]))
                fill[take] = fixed[k][take]
        elif value_source == "mean":
            fill = np.nanmean(fixed, axis=0).astype(np.float32)
        else:
            raise ValueError("value_source must be 'first' or 'mean'")

        # For each raster, wherever it is nan AND mismatch cell, replace with fill
        for k in range(fixed.shape[0]):
            to_fill = np.isnan(fixed[k]) & mismatch
            fixed[k][to_fill] = fill[to_fill]
    else:
        raise ValueError("policy must be 'prefer_nodata' or 'prefer_value'")

    # Sanity check: after fix, mismatch should be gone
    is_nan2 = np.isnan(fixed)
    mismatch2 = is_nan2.any(axis=0) & (~is_nan2.all(axis=0))
    print(f"Mismatch after fix: {int(mismatch2.sum())}")

    # Write outputs
    if dry_run:
        print("Dry-run enabled: not writing files.")
        return

    for k, src in enumerate(raster_paths):
        src = Path(src)
        out_path = output_dir / src.name
        write_asc(out_path, fixed[k], headers[k], nodata)
        print(f"Wrote: {out_path}")


# ----------------------------
# USER INPUT (interactive)
# ----------------------------
# Put your rasters here (edit paths as needed)
rasters_to_fix = [
    "data/moz/moz_initialpopulation.asc",
    "data/moz/moz_beta.asc",
    "data/moz/moz_treatmentseeking.asc",
    "data/moz/moz_traveltime.asc",
    "data/moz/moz_districts.asc",
]

print("Choose policy for mismatch cells (some rasters NODATA, others valid):")
print("  1) prefer_nodata  -> set ALL rasters to NODATA at mismatch cells (intersection mask).")
print("  2) prefer_value   -> fill NODATA cells using values from non-NODATA rasters (union mask).")
choice = input("Enter 1 or 2: ").strip()

if choice == "1":
    policy = "prefer_nodata"
    value_source = "first"
elif choice == "2":
    policy = "prefer_value"
    print("Choose fill source for prefer_value:")
    print("  a) first -> take first non-NODATA value among rasters")
    print("  b) mean  -> take mean of non-NODATA values among rasters")
    c2 = input("Enter a or b (default a): ").strip().lower()
    value_source = "mean" if c2 == "b" else "first"
else:
    raise ValueError("Invalid choice. Enter 1 or 2.")

out_dir = input("Output directory (default: data/moz/fixed): ").strip()
out_dir = out_dir if out_dir else "data/moz/fixed"
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

fix_raster_masks(
    rasters_to_fix,
    out_dir,
    policy=policy,
    value_source=value_source,
    dry_run=False,
)


Choose policy for mismatch cells (some rasters NODATA, others valid):
  1) prefer_nodata  -> set ALL rasters to NODATA at mismatch cells (intersection mask).
  2) prefer_value   -> fill NODATA cells using values from non-NODATA rasters (union mask).
Choose fill source for prefer_value:
  a) first -> take first non-NODATA value among rasters
  b) mean  -> take mean of non-NODATA values among rasters
Loaded 5 rasters: shape=368x234, NODATA=-9999.0
Mismatch cells (some NODATA, some valid): 29206
Mismatch after fix: 0
Wrote: data/moz/fixed/moz_initialpopulation.asc
Wrote: data/moz/fixed/moz_beta.asc
Wrote: data/moz/fixed/moz_treatmentseeking.asc
Wrote: data/moz/fixed/moz_traveltime.asc
Wrote: data/moz/fixed/moz_districts.asc


/tmp/ipykernel_3975544/2740603501.py:127: RuntimeWarning: Mean of empty slice
  fill = np.nanmean(fixed, axis=0).astype(np.float32)
